In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yazanalshuaibi/gas-furnace")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
df = pd.read_csv(os.path.join(path,"gas-furnace.csv"))

In [ ]:
df = df[["CO2"]]
df.head()

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.reset_index(drop=True, inplace=True)
df.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(df["CO2"], label="CO2")
plt.legend()
plt.title("CO2 Time Series")
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure()
plot_acf(df["CO2"], lags=40)
plt.title("Autocorrelation of CO2")
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(df["CO2"], lags=30)
plt.title("Partial Autocorrelation of CO2")
plt.show()

In [ ]:
def time_split(df, test_size=0.2):
    df.index = pd.date_range("2026-01-01", periods=len(df), freq="1s")
    n = len(df)
    split = int((1 - test_size) * n)
    return df.iloc[:split], df.iloc[split:]

In [ ]:
train, test = time_split(df)

In [ ]:
train

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima = ARIMA(
    train["CO2"],
    order=(3, 0, 1),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)
arima_fit = arima.fit()

fit_ext = arima_fit.apply(df["CO2"])

pred = fit_ext.get_prediction(start=int(0.8 * len(df)), end=len(df)-1, dynamic=False)

yhat_arima = pred.predicted_mean.to_numpy()

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(test, yhat_arima)
print(f"ARIMA(3,0,1) MAE: {mae:.4f}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(range(len(test)), test, label="Actual CO2")
plt.plot(range(len(yhat_arima)), yhat_arima, label="ARIMA(3,0,1) Prediction")
plt.legend()
plt.title("ARIMA(3,0,1) - Actual vs Predicted")
plt.xlabel("Time Step")
plt.ylabel("CO2")
plt.show()